# XM655 calibration algorithm measurements

Transmit a tone, capture it on all 16 ADCs, keep the channels that matter, and
sweep a list of beam weights.

**TX and RX are started by the same trigger.** In the PL the player enable is
`dac_enable OR trig_cap`, so holding `dac_enable` low hands the enable to
`trig_cap`: one rising edge restarts the waveform at sample 0 and arms the
capture on the same clock. Every burst then begins at the same point in the
waveform and the global phase does not move between bursts.

Reconfiguring the DACs between bursts is safe. `configure_dacs()` does call
`ResetNCOPhase()`, but `EVENT_SRC` is `EVNT_SRC_SYSREF`, so that reset lands on
a SYSREF edge and the DAC-to-ADC phase relationship survives it. Measured
repeatability is better than 0.4 deg.

The hardware always hands back all 16 channels, but the ones that are not in
`FF_ADC` (far field) or `NF_ADC` (near field) are thrown away immediately, in
the raw int16 stream, before anything is converted or plotted. From section 6
onwards a capture only has the rows in `KEEP`, and the far and near field rows
are saved to two separate files.

Run the cells top to bottom.

## 1. Setup

Load the bitstream. Takes a few seconds and only needs to happen once per
kernel - and only one kernel at a time may hold the overlay.

In [ ]:
import json
import os
import time

import numpy as np
import matplotlib.pyplot as plt

from lib.mts import doaMtsOverlay
from lib.config_parser import load_config
from lib.common_functions import capture_aligned
from lib.common_functions import create_tone_samples
from lib.common_functions import create_zc_chirp_samples
from lib.common_functions import save_json_params
from lib.common_functions import snap_tone_to_fft_bin
from lib.common_functions import write_tone_to_players

CFG = load_config()
overlay = doaMtsOverlay("mts.bit")

## 2. Settings

Split in two: RF configuration (converter frequencies, capture size, per-DAC
phase/gain, sweep tables) and baseband signal configuration (what waveform
actually gets played). Nothing below section 2 needs editing.

The NCO sets the RF frequency and is per *tile*, so all four DACs in a tile
share one waveform and differ only in phase and gain - which is what steers the
beam.

The ADC samples at 2.5 GSPS, so a 4900 MHz signal folds down to
`5000 - 4900 = 100 MHz`. Rule of thumb: **`ADC_NCO = 5000 - DAC_NCO`**, and the
tone comes back at its own frequency. Do not go much above 4900 MHz - the
receive mixer also makes an image twice the fold away, and if the fold is small
that image lands in band and ruins the signal.

Phase and gain are written as **one row per capture, one column per DAC of the
transmit tile**. The sweep table uses the same shape, so a row means the same
thing everywhere.

`FF_ADC` and `NF_ADC` are indices into the full 16-channel capture and decide
what gets written to disk: the far-field rows go to `rx_ff.npy`, the near-field
rows to `rx_nf.npy`, in the order listed. Everything else is dropped after the
plots. `FF_ANGLE` is recorded in `params.json` and does nothing else - it is
there so a capture remembers which far-field angle it was taken at.

In [ ]:
# --- rf ---
DAC_NCO  = CFG["rf"]["dac_nco"]      # MHz, per tile -> TX lands at 4900
DAC_ZONE = CFG["rf"]["dac_zone"]     # Nyquist zone, per tile
ADC_NCO  = CFG["rf"]["adc_nco"]      # MHz, per tile = 5000 - DAC_NCO
ADC_ZONE = CFG["rf"]["adc_zone"]     # the fold is in an even zone

# --- one capture ---
N_CAP     = CFG["capture"]["n_cap"]  # samples per channel
DAC_PHASE = [0, 0, 0, 0]             # degrees, one per transmit column
DAC_GAIN  = [1, 0, 1, 0]             # 0..1, one per transmit column

# --- which ADCs get kept, from config/parameters.json ---
FF_ADC   = CFG["systems"]["adcs_1"]  # far field, the other node  -> rx_ff.npy
NF_ADC   = CFG["systems"]["adcs_0"]  # near field, our own ears   -> rx_nf.npy

# --- far field angle, written to params.json, no effect on the capture ---
FF_ANGLE = 0.0             # degrees

# --- alignment check ---
N_BURSTS = 50
REF_CH   = FF_ADC[0]       # ADC to watch

# --- switches ---
PLOT_SPECTRUM   = True     # section 6
PLOT_TIME       = True     # section 6
SAVE_CAPTURE    = False    # section 7
RUN_SWEEP       = False    # section 8
RUN_PHASE_CHECK = False    # section 9

# --- sweep table: one row per capture, same columns as DAC_PHASE ---
SWEEP_PHASE = [
    [  0, 0,   0.00, 0], [  0, 0,   0.00, 0],
    [  0, 0, 179.99, 0], [  0, 0, 179.99, 0],
    [  0, 0,  45.00, 0], [  0, 0,  45.00, 0],
    [ 90, 0,  30.00, 0], [ 90, 0,  30.00, 0],
    [ 10, 0, -15.00, 0], [ 10, 0, -15.00, 0],
    [ 10, 0, 164.99, 0], [ 10, 0, 164.99, 0],
    [ 10, 0,  30.00, 0], [ 10, 0,  30.00, 0],
    [100, 0,  15.00, 0], [100, 0,  15.00, 0],
]

SWEEP_GAIN = [
    [1.0, 0, 1.0, 0], [1.0, 0, 1.0, 0],
    [1.0, 0, 1.0, 0], [1.0, 0, 1.0, 0],
    [1.0, 0, 1.0, 0], [1.0, 0, 1.0, 0],
    [1.0, 0, 1.0, 0], [1.0, 0, 1.0, 0],
    [0.9, 0, 0.8, 0], [0.9, 0, 0.8, 0],
    [0.9, 0, 0.8, 0], [0.9, 0, 0.8, 0],
    [0.9, 0, 0.8, 0], [0.9, 0, 0.8, 0],
    [0.9, 0, 0.8, 0], [0.9, 0, 0.8, 0],
]

### Baseband signal configuration

What actually gets played, independent of the RF settings above.

- **`SIG_TYPE = "tone"`** - a single harmonic at `TONE_MHZ`, offset down from
  the tile NCO. The frequency gets snapped to a whole number of cycles in the
  capture window in section 3, otherwise it lands between FFT bins.
- **`SIG_TYPE = "chirp"`** - a narrowband Zadoff-Chu-style CAZAC chirp: constant
  amplitude, quadratic phase, instantaneous frequency sweeping linearly from 0
  to `ZC_BW_MHZ` across the capture window.

In [ ]:
# --- baseband signal ---
SIG_TYPE  = "tone"                     # "tone" or "chirp"
TONE_MHZ  = CFG["signal"]["tone_0_mhz"]  # baseband tone, offset down from the NCO
AMP       = CFG["signal"]["amp"]       # 14 bit DAC: +16383 / -16384
ZC_BW_MHZ = 1.0                        # chirp only: instantaneous freq sweeps 0 -> ZC_BW_MHZ

## 3. Board mapping

Fixed by the bitstream and by how this board is wired - change it only if the
hardware changes.

`DAC_MAP` is the one that bites. The `overlay.dac[]` index is **not** the RF
channel label: on this board the tile that reaches the antennas is tile 2, so
transmit column 0 is `overlay.dac[8]` and column 2 is `overlay.dac[10]`. DAC 8
lands on ADC 0 and 4, DAC 10 on ADC 2 and 6, about 35 dB above everything else.
Write the weights to the wrong DACs and the capture is only crosstalk - it
still looks like a signal, but the phase is noise.

To re-measure it on another board, un-mute one DAC at a time
(`overlay.dac[d].QMCSettings['GainCorrectionFactor']`) and see where the energy
lands.

The DAC interpolates and the ADC decimates by 10, so both baseband rates come
from the converter rates, not from the fabric clock.

`KEEP` is the union of `FF_ADC` and `NF_ADC`, sorted, and it is the row order
of every capture from section 6 on. `row()` turns an ADC number into its row,
so the rest of the notebook can keep talking in ADC numbers.

In [ ]:
DAC_SR  = CFG["board"]["dac_sr"]     # DAC baseband rate = 10 GSPS / 10 (C2R eats one x2)
ADC_SR  = CFG["board"]["adc_sr"]     # ADC baseband rate = 2.5 GSPS / 10
N_CH    = CFG["board"]["n_ch"]       # RF channels on the XM655
DAC_MAP = CFG["systems"]["dacs_0"]   # transmit column -> overlay.dac index

# trig_cap has to stay high for a whole capture window - the DAC only plays
# while the enable is high, so releasing early leaves the tail silent
TRIG_HOLD_S = CFG["capture"]["trig_hold_s"]

# the channels worth carrying past the capture, and the row order of every
# capture from section 6 on
KEEP = sorted(set(FF_ADC) | set(NF_ADC))

if not KEEP:
    raise ValueError("FF_ADC and NF_ADC are both empty - nothing to capture")
if not set(KEEP) <= set(range(N_CH)):
    raise ValueError("config/parameters.json lists ADCs %s, outside 0..%d"
                     % (sorted(set(KEEP) - set(range(N_CH))), N_CH - 1))
if REF_CH not in KEEP:
    raise ValueError("REF_CH %d is not in FF_ADC %s or NF_ADC %s"
                     % (REF_CH, FF_ADC, NF_ADC))


def validate_transmit_width(name, table):
    """One column per transmit DAC, or say which two numbers disagree.

    DAC_MAP comes from config/parameters.json while the tables are written here,
    so the two drift apart silently: a short config drops the last column of every
    row, a long one only fails once the sweep is already writing captures.
    """
    rows = table if isinstance(table[0], (list, tuple)) else [table]
    for index, row in enumerate(rows):
        if len(row) != len(DAC_MAP):
            raise ValueError(
                "%s row %d has %d column(s), but config/parameters.json lists "
                "%d transmit DAC(s) %s - the table needs one column per DAC"
                % (name, index, len(row), len(DAC_MAP), DAC_MAP))


validate_transmit_width("DAC_PHASE", DAC_PHASE)
validate_transmit_width("DAC_GAIN", DAC_GAIN)
validate_transmit_width("SWEEP_PHASE", SWEEP_PHASE)
validate_transmit_width("SWEEP_GAIN", SWEEP_GAIN)


def to_dacs(row):
    """One transmit row -> one value per DAC, everything unmapped muted."""
    full = [0.0] * N_CH
    for col, dac in enumerate(DAC_MAP):
        full[dac] = float(row[col])
    return full


def row(ch):
    """Row of a trimmed capture holding ADC `ch`."""
    return KEEP.index(ch)


def ch_label(ch):
    """ADC number plus which list it came from, for plot titles."""
    tag = "/".join(name for name, lst in (("FF", FF_ADC), ("NF", NF_ADC))
                   if ch in lst)
    return "ADC %d (%s)" % (ch, tag)

## 4. Generate signal

Build the baseband waveform for `SIG_TYPE`. Runs after Board mapping because it
needs `DAC_SR` and the player length, and before Transmit because that's just
where the result gets written into the tile memories.

In [ ]:
n_samples = overlay.dac0_player.shape[0] // 2

if SIG_TYPE == "tone":
    # the tone must fit a whole number of cycles in the capture, otherwise it
    # lands between FFT bins and the measured phase is meaningless
    TONE_MHZ = snap_tone_to_fft_bin(TONE_MHZ, ADC_SR, N_CAP)
    TONE_HZ = TONE_MHZ * 1e6
    baseband = create_tone_samples(n_samples, DAC_SR, TONE_HZ, AMP)
elif SIG_TYPE == "chirp":
    baseband = create_zc_chirp_samples(n_samples, DAC_SR, ZC_BW_MHZ * 1e6, AMP)
else:
    raise ValueError("SIG_TYPE must be 'tone' or 'chirp', got %r" % SIG_TYPE)

## 5. Transmit

Push the settings into the DACs, copy the signal built in section 4 into all
four tile memories, then **arm without running**: `dacs_off()` leaves
`dac_enable` low so `trig_cap` owns the player enable.

In [ ]:
overlay.d_centre_freq  = DAC_NCO
overlay.d_nyquist_zone = DAC_ZONE
overlay.d_phases       = to_dacs(DAC_PHASE)
overlay.d_gain         = to_dacs(DAC_GAIN)
overlay.configure_dacs()

write_tone_to_players(overlay, baseband)

overlay.dacs_off()
overlay.da = 2

print("DACs armed (%s) at %.3f MHz - idle until triggered"
      % (SIG_TYPE, DAC_NCO[0] - (TONE_HZ / 1e6 if SIG_TYPE == "tone" else 0)))

## 6. Receive

Each of the 16 ADCs has its own antenna input. Set the channel list **and**
`data_size` before capturing, otherwise the driver quietly returns only the
first 4 channels, and tune before `configure_adcs()` or the settings land one
run late.

`capture_aligned()` replaces `get_custom_data_xm655()`, which would fire its own
trigger and undo the alignment.

`raw_to_iq()` drops the unused channels while the capture is still interleaved
int16: it reshapes the stream into frames, picks out the I/Q lanes of `KEEP`,
and only then builds complex numbers. The channels we do not want never become
floats, so the array that comes out is `len(KEEP)` rows, not 16.

In [ ]:
def raw_to_iq(raw, keep=None, n_ch=N_CH):
    """Flat interleaved int16 -> (len(keep), N) complex.

    The unused channels are dropped here, on the int16 frames, so they are
    never widened to float - the whole point of trimming this early.
    """
    keep = KEEP if keep is None else keep
    raw = np.asarray(raw, dtype=np.int16).ravel()
    lanes = n_ch * 2
    if raw.size % lanes:
        raise ValueError("%d samples is not divisible by %d" % (raw.size, lanes))

    frames = raw.reshape(-1, lanes)
    i = frames[:, [2 * ch for ch in keep]].T
    q = frames[:, [2 * ch + 1 for ch in keep]].T
    return i.astype(np.float64) + 1j * q


def tone(iq, ch=REF_CH):
    """Complex amplitude of the strongest bin on one ADC."""
    F = np.fft.fft(iq[row(ch)])
    return F[np.abs(F).argmax()]

In [ ]:
overlay.active_rf_channels = list(range(N_CH))
overlay.channels  = N_CH * 2                    # I and Q per channel
overlay.data_size = N_CAP * overlay.channels

overlay.centre_freq  = ADC_NCO
overlay.nyquist_zone = ADC_ZONE
overlay.phases       = [0] * N_CH
overlay.configure_adcs()

iq = raw_to_iq(capture_aligned(overlay, TRIG_HOLD_S))
print("captured", iq.shape)

## 7. Plot

Two grids, one panel per kept channel: the spectrum, so you can see where the
energy is, and the time domain, so you can see what it looks like. Titles carry
the real ADC number and whether it is a far or near field channel, so the plots
stay readable when `FF_ADC` / `NF_ADC` change. All dB are relative to the
strongest kept channel, so a quiet channel looks quiet.

In [ ]:
def grid(n, sharey=False):
    """A panel per kept channel, at most four across. Spares are hidden."""
    cols = min(4, n)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 2.6 * rows),
                             sharex=True, sharey=sharey, squeeze=False)
    for ax in axes.flat[n:]:
        ax.axis("off")
    return fig, list(axes.flat[:n])


def plot_spectrum(iq, keep=None):
    keep = KEEP if keep is None else keep
    n = iq.shape[1]
    f = np.fft.fftshift(np.fft.fftfreq(n, 1 / ADC_SR)) / 1e6
    S = np.abs(np.fft.fftshift(np.fft.fft(iq * np.hanning(n), axis=1), axes=1))
    S = 20 * np.log10(S / S.max() + 1e-12)

    fig, axes = grid(len(keep), sharey=True)
    for ch, s, ax in zip(keep, S, axes):
        ax.plot(f, s)
        ax.set_title("%s   peak %+.2f MHz" % (ch_label(ch), f[s.argmax()]),
                     fontsize=9)
        ax.grid(True)
    axes[0].set_ylim(-80, 5)
    fig.supxlabel("MHz")
    fig.supylabel("dB (relative to strongest kept channel)")
    plt.tight_layout()
    plt.show()


def plot_iq(iq, keep=None, max_samples=400):
    keep = KEEP if keep is None else keep
    n = min(max_samples, iq.shape[1])
    fig, axes = grid(len(keep))
    for ch, chan, ax in zip(keep, iq, axes):
        ax.plot(chan[:n].real, label="I")
        ax.plot(chan[:n].imag, label="Q")
        ax.set_title(ch_label(ch))
        ax.grid(True)
    axes[0].legend()
    fig.supxlabel("Sample")
    fig.supylabel("ADC value")
    plt.tight_layout()
    plt.show()


if PLOT_SPECTRUM:
    plot_spectrum(iq)
if PLOT_TIME:
    plot_iq(iq)

## 8. Save

One folder per capture, holding five files:

| file | what it is |
|---|---|
| `tx_signal.npy` | the baseband waveform we sent, complex, one vector |
| `tx_weights.npy` | one complex weight per DAC - `gain * exp(j*phase)` |
| `rx_ff.npy` | the far-field rows, complex, shape `(len(FF_ADC), N)` |
| `rx_nf.npy` | the near-field rows, complex, shape `(len(NF_ADC), N)` |
| `params.json` | every setting from section 2, plain text |

Row `i` of `rx_ff.npy` is ADC `FF_ADC[i]`, and the same for the near field, so
`params.json` carries both lists - without them the rows are anonymous. The
channels outside those two lists were dropped back in section 6 and never reach
here.

Everything is `complex64` - the samples are 14 bit integers, so no precision is
lost and the files are half the size. A sweep fires several captures inside the
same second, so the folder name is passed in rather than taken from the clock.

Set `SAVE_CAPTURE = True` in section 2 to save the single capture above.

In [ ]:
def select(iq, chans):
    """Rows of a trimmed capture for the ADCs asked for, in that order."""
    return np.asarray(iq)[[row(ch) for ch in chans]]


def save_capture(iq, folder="output/captures", name=None, phase=None, gain=None):
    """Write one capture into its own folder, far and near field separately."""
    phase = to_dacs(DAC_PHASE) if phase is None else phase
    gain  = to_dacs(DAC_GAIN) if gain is None else gain

    path = os.path.join(folder, name or time.strftime("%Y%m%d_%H%M%S"))
    os.makedirs(path, exist_ok=True)

    tx_signal = overlay.signal[0::2] + 1j * overlay.signal[1::2]
    tx_weights = np.array(gain) * np.exp(1j * np.deg2rad(phase))

    np.save(os.path.join(path, "tx_signal.npy"),  tx_signal.astype(np.complex64))
    np.save(os.path.join(path, "tx_weights.npy"), tx_weights.astype(np.complex64))
    np.save(os.path.join(path, "rx_ff.npy"), select(iq, FF_ADC).astype(np.complex64))
    np.save(os.path.join(path, "rx_nf.npy"), select(iq, NF_ADC).astype(np.complex64))

    save_json_params(path, {"dac_nco": DAC_NCO, "dac_zone": DAC_ZONE,
                            "adc_nco": ADC_NCO, "adc_zone": ADC_ZONE,
                            "dac_phase": list(phase), "dac_gain": list(gain),
                            "dac_map": DAC_MAP, "ff_adc": list(FF_ADC),
                            "nf_adc": list(NF_ADC), "ff_angle": FF_ANGLE,
                            "sig_type": SIG_TYPE,
                            "tone_hz": TONE_HZ if SIG_TYPE == "tone" else None,
                            "zc_bw_mhz": ZC_BW_MHZ if SIG_TYPE == "chirp" else None,
                            "amp": AMP, "dac_sr": DAC_SR, "adc_sr": ADC_SR})

    return path


if SAVE_CAPTURE:
    print("saved ->", save_capture(iq))

## 9. Sweep

One aligned capture per row of `SWEEP_PHASE` / `SWEEP_GAIN`, all written under
one parent folder:

```
output/captures/sweep_<timestamp>/
    row00_p0-0_g1.00-1.00/
    row01_p0-0_g1.00-1.00/
    ...
```

Rows with identical weights must come back to the same phase, so the sweep
checks that itself at the end. A large spread means either the weights are
going to unconnected DACs (check `DAC_MAP`) or the trigger alignment has
stopped working - section 10 tells the two apart.

Set `RUN_SWEEP = True` in section 2 to turn this on.

In [ ]:
def run_sweep(phases, gains, folder="output/captures"):
    """One aligned capture per row, all of them saved under one folder."""
    if len(phases) != len(gains):
        raise ValueError("phase and gain tables must have the same length")

    run = os.path.join(folder, "sweep_" + time.strftime("%Y%m%d_%H%M%S"))
    repeats = {}

    for i, (p_row, g_row) in enumerate(zip(phases, gains)):
        p_full, g_full = to_dacs(p_row), to_dacs(g_row)
        overlay.d_phases = p_full
        overlay.d_gain   = g_full
        overlay.configure_dacs()

        iq = raw_to_iq(capture_aligned(overlay, TRIG_HOLD_S))
        name = "row%02d_p%g-%g_g%.2f-%.2f" % (i, p_row[0], p_row[2],
                                              g_row[0], g_row[2])
        print("saved ->", save_capture(iq, folder=run, name=name,
                                       phase=p_full, gain=g_full))
        repeats.setdefault((tuple(p_row), tuple(g_row)), []).append(tone(iq))

    overlay.dacs_off()

    worst = max((np.abs(np.rad2deg(np.angle(np.array(z) / z[0]))).max()
                 for z in repeats.values() if len(z) > 1), default=0.0)
    print("\nswept %d settings -> %s" % (len(phases), run))
    print("largest spread between repeats of the same setting: %.2f deg" % worst)
    if worst > 5:
        print("  -> too big, check DAC_MAP and the signal level")
    return run


if RUN_SWEEP:
    run_sweep(SWEEP_PHASE, SWEEP_GAIN)

## 10. Check the alignment

Capture the same tone many times without touching anything and look at the
phase on one channel. If the trigger is doing its job the phase should not move.

- **flat line** - aligned, nothing more to do
- **a few discrete levels** - the trigger is landing on different clock edges;
  retune so `DAC_NCO - ADC_NCO` is a multiple of 250 MHz
- **scattered** - check the amplitude first. A signal 30 dB down is crosstalk,
  and its phase is noise; that is a `DAC_MAP` problem, not an alignment one.

Set `RUN_PHASE_CHECK = True` in section 2 to turn this on.

In [ ]:
if RUN_PHASE_CHECK:
    bursts = [tone(raw_to_iq(capture_aligned(overlay, TRIG_HOLD_S))) for _ in range(N_BURSTS)]

    phase = np.rad2deg(np.unwrap(np.angle(bursts)))
    phase -= phase[0]

    print("peak-to-peak %.2f deg,  std %.2f deg" % (np.ptp(phase), phase.std()))
    print("amplitude %.3g  (crosstalk is ~30 dB below the real signal)"
          % np.abs(bursts).mean())

    plt.figure(figsize=(10, 3))
    plt.plot(phase, "o-")
    plt.xlabel("burst")
    plt.ylabel("phase drift (deg)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## 11. Stop

Switch the transmitter off when you are done.

In [11]:
overlay.dacs_off()